# 08.5 mini Codec-LM：在 codec token 上训练语言模型

MusicGen 的核心想法是在神经 codec 的离散 token 上建模。本 Notebook 使用 FMA small 作为音频来源，要求先生成 EnCodec token cache，没有 token cache 时不会换成随机 token fallback。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Audio, display

from _common.audio_io import save_audio
from _common.dataset_registry import check_required_assets
from _common.device_utils import choose_device
from _common.paths import portable_path
from _common.plotting import finish_figure, setup_plot_style
from codec.build_token_cache import build_fma_token_cache, resolve_fma_audio_paths
from codec.encodec_adapter import check_encodec
from codec.teaching_token_cache import TeachingCodecConfig, build_teaching_fma_token_cache, teaching_tokens_to_audio
from codec.token_visualization import plot_token_matrix
from codec_lm.model import MiniCodecLM, generate_tokens
from codec_lm.token_dataset import find_token_files, flatten_codec_tokens
from codec_lm.train import train_codec_lm

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_AUDIO = ROOT / "output_audio" / "08_5"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
ENCODEC_TOKEN_CACHE = ROOT / "outputs" / "generated" / "codec_tokens"
TEACHING_TOKEN_CACHE = ROOT / "outputs" / "generated" / "teaching_codec_tokens"
CHECKPOINT_DIR = ROOT / "outputs" / "checkpoints" / "codec_lm_08_5"
FMA_MANIFEST = ROOT / "data_manifests" / "fma_small_subset.csv"
for path in [OUTPUT_FIGURES, OUTPUT_AUDIO, OUTPUT_TABLES, ENCODEC_TOKEN_CACHE, TEACHING_TOKEN_CACHE, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()
DEVICE = choose_device(os.getenv("CHAPTER08_DEVICE", "auto"))
print("selected device:", DEVICE)

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
check_required_assets(
    ["fma_small"],
    message="08_5 requires FMA small audio and metadata for mini Codec-LM training.",
    stop=True,
)
if not FMA_MANIFEST.exists():
    print("Missing FMA small subset manifest:", rel(FMA_MANIFEST))
    print("Build it with: python data_manifests/build_chapter08_manifests.py --fma-limit 512")
    raise FileNotFoundError(FMA_MANIFEST)
fma_examples = resolve_fma_audio_paths(audio_manifest_csv=FMA_MANIFEST, limit=5)
print("Using FMA manifest:", rel(FMA_MANIFEST))
print("FMA example files:")
for path in fma_examples:
    print("-", rel(path))


In [ ]:
encodec_token_files = find_token_files(ENCODEC_TOKEN_CACHE)
teaching_token_files = find_token_files(TEACHING_TOKEN_CACHE)
encodec_status = check_encodec()
print("cached EnCodec token files:", len(encodec_token_files))
print("cached teaching token files:", len(teaching_token_files))
print("encodec status:", encodec_status)


In [ ]:
# BUILD_TOKEN_CACHE 默认 False：构建 EnCodec 缓存需先下载模型权重并逐条编码 FMA 片段，耗时较长。
# 为避免运行 Notebook 时误触发长任务而默认关闭，缺缓存时自动回退，训练流程不受影响。
# 要走神经 codec 路径时改为 True 重跑（需 encodec 可用）。
BUILD_TOKEN_CACHE = False

if not encodec_token_files and encodec_status.available and BUILD_TOKEN_CACHE:
    build_fma_token_cache(
        token_dir=ENCODEC_TOKEN_CACHE,
        manifest_csv=OUTPUT_TABLES / "08_5_token_cache_manifest.csv",
        audio_manifest_csv=FMA_MANIFEST,
        limit=8,
        model_name="24khz",
        bandwidth=6.0,
        device=DEVICE,
        save_reconstruction=False,
    )
    encodec_token_files = find_token_files(ENCODEC_TOKEN_CACHE)

if not encodec_token_files and not teaching_token_files:
    print("No EnCodec token cache found. Building deterministic teaching tokens from FMA small.")
    print("For the neural codec path, install encodec, then run:")
    print("python -m codec.build_token_cache --audio-manifest data_manifests/fma_small_subset.csv --limit 32 --device auto")
    print("Install command:", encodec_status.next_action)
    build_teaching_fma_token_cache(
        token_dir=TEACHING_TOKEN_CACHE,
        manifest_csv=OUTPUT_TABLES / "08_5_teaching_token_cache_manifest.csv",
        audio_manifest_csv=FMA_MANIFEST,
        limit=16,
        config=TeachingCodecConfig(duration_sec=4.0, n_mels=16, vocab_size=256),
    )
    teaching_token_files = find_token_files(TEACHING_TOKEN_CACHE)

if encodec_token_files:
    token_files = encodec_token_files
    token_cache_dir = ENCODEC_TOKEN_CACHE
    token_source = "encodec"
    vocab_size = 1024
else:
    token_files = teaching_token_files
    token_cache_dir = TEACHING_TOKEN_CACHE
    token_source = "teaching_logmel"
    vocab_size = 256

print("Using token source:", token_source)
for path in token_files[:5]:
    print("-", rel(path))


In [ ]:
if token_files:
    config = {
        "seed": 8,
        "device": DEVICE,
        "data": {
            "source": "token_cache",
            "token_cache_dir": rel(token_cache_dir),
            "token_source": token_source,
            "max_files": 32,
        },
        "model": {
            "vocab_size": vocab_size,
            "d_model": 64,
            "n_heads": 4,
            "n_layers": 2,
            "max_length": 128,
        },
        "training": {
            "batch_size": 4,
            "epochs": 3,
            "learning_rate": 0.0005,
            "max_steps_per_epoch": 8,
        },
        "outputs": {
            "checkpoint_dir": rel(CHECKPOINT_DIR),
            "history_csv": rel(OUTPUT_TABLES / "08_5_codec_lm_history.csv"),
        },
    }
    history = train_codec_lm(config)
    history_df = pd.DataFrame(history)
    display(history_df)
else:
    history_df = pd.DataFrame()


**小型编解码器语言模型训练曲线**

三个轮次里交叉熵与困惑度同步下降，说明训练闭环在工作；幅度受限于合计 24 次参数更新。


In [ ]:
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", color="0.15")
    axes[0].set_title("mini Codec-LM 损失")
    axes[0].set_xlabel("轮次")
    axes[0].set_ylabel("交叉熵")
    axes[1].plot(history_df["epoch"], history_df["perplexity"], marker="o", color="0.35")
    axes[1].set_title("困惑度")
    axes[1].set_xlabel("轮次")
    finish_figure(fig, OUTPUT_FIGURES / "08_5_codec_lm_training_curves.png")
    plt.show()
else:
    print("Skip training curves because no token cache is available.")


下面从一段真实音频的 token 中取出前 16 个作为种子——按帧展平之后，这 16 个值恰好覆盖一个时间帧的 16 个 mel 频带——再让模型自回归生成 512 个新 token（对应 32 帧），最后把生成的 token 序列还原成音频。当前环境使用教学 token，还原路径是先把 token 反映射成 log-mel 频谱，再用 Griffin-Lim 迭代估计相位、合成波形；如果本地装有 EnCodec 并构建了神经 codec token 缓存，同一份模型输出会改由神经编解码器解码。

播放器里的声音只有约 0.7 秒，听感接近噪声并不意外，两层因素叠加在一起。教学编码器本身很粗糙，16 个频带、256 级量化抹掉了大部分频谱细节，相位信息完全丢失，Griffin-Lim 只能用 8 次迭代近似找回波形，即使输入真实 token，重建出来的声音也只是勉强可辨。更主要的限制在模型端：种子只提供了一帧的上下文，而这个小型 Transformer 总共只更新了 24 次参数，生成的 token 接近随机，解码后自然谈不上音乐性。本 Notebook 的目标是走通 token 上的训练闭环——数据集、教师强制、交叉熵损失、自回归采样——而非得到可听的生成结果；要判断这个演示模型学到了什么，看下方的 token 矩阵图比听音频更直接。


**小型编解码器语言模型生成的离散音频 token 序列**

前 16 个 token 是来自真实音频的种子，其余由模型自回归生成，灰度对应 token 编号。


In [ ]:
if token_files:
    checkpoint = torch.load(CHECKPOINT_DIR / "last.pt", map_location="cpu")
    model_cfg = config["model"]
    model = MiniCodecLM(**model_cfg)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE)
    model.eval()
    seed = flatten_codec_tokens(torch.load(token_files[0], map_location="cpu"))[:16] % model.vocab_size
    seed = seed.to(DEVICE)
    num_new = 16 * 32 if token_source == "teaching_logmel" else 64
    generated = generate_tokens(model, seed.unsqueeze(0), num_new_tokens=num_new, temperature=0.95)
    fig = plot_token_matrix(
        generated.detach().cpu().squeeze(0).numpy()[None, :],
        OUTPUT_FIGURES / "08_5_codec_lm_generated_tokens.png",
    )
    plt.show()
    print("generated token shape:", tuple(generated.shape))
    if token_source == "teaching_logmel":
        decode_config = TeachingCodecConfig(duration_sec=4.0, n_mels=16, vocab_size=256)
        generated_audio = teaching_tokens_to_audio(generated.detach().cpu().squeeze(0), decode_config, n_iter=8)
        save_audio(OUTPUT_AUDIO / "codec_lm_teaching_generated.wav", generated_audio, decode_config.sample_rate)
        display(Audio(str(OUTPUT_AUDIO / "codec_lm_teaching_generated.wav")))
else:
    print("Skip token generation because no trained mini Codec-LM is available.")


In [ ]:
print("08_5 status:", f"trained mini Codec-LM on {token_source} tokens" if token_files else "waiting for token cache")
for path in sorted(OUTPUT_FIGURES.glob("08_5_*.png")):
    print("-", rel(path))
for path in sorted(OUTPUT_AUDIO.glob("08_5_*.wav")) + sorted(OUTPUT_AUDIO.glob("codec_lm_*.wav")):
    print("-", rel(path))
